# FSPS + MAF on 100,000 COSMOS2020 galaxies

This is the large-catalog FSPS workflow: a continuity SFH, explicit catalog-derived
noise, 300,000 prior simulations, one uncertainty-conditioned MAF, and memory-safe
posterior summaries for 100,000 observed SEDs. COSMOS2020/LePhare values are
comparison estimates, not truth.


In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import sys
import numpy as np

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "composed").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
DATA_PATH = REPO_ROOT / "notebooks/tutorials/data/cosmos2020_ugrizYJH_100k.npz"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run 00_prepare_cosmos2020_ugrizYJH.ipynb first.")

with np.load(DATA_PATH, allow_pickle=False) as catalog:
    flux = catalog["flux_maggies"]
    sigma = catalog["sigma_maggies"]
    z_reference = catalog["lp_zbest"]
    mass_reference = catalog["lp_mass_med"]
    sfr_reference = catalog["lp_sfr_med"]
    dust_reference = catalog["lp_dust"]
    BAND_NAMES = tuple(catalog["band_names"].astype(str))
    FILTER_NAMES = tuple(catalog["sedpy_filter_names"].astype(str))
    SINGLE_OBJECT = int(catalog["single_object_position"])

print(f"Loaded {flux.shape[0]:,} galaxies in {flux.shape[1]} bands")
print("Single tutorial object:", SINGLE_OBJECT, "z_ref=", z_reference[SINGLE_OBJECT])

from sedpy.observate import load_filters
from composed.filters import FilterSet

filters = FilterSet(load_filters(list(FILTER_NAMES)), names=BAND_NAMES)


## Model and priors


In [ ]:
from composed import (
    ConditionalCatalogNoise, ContinuitySFH, Gaussian, ParameterSpace,
    Problem, SEDDataset, StudentTPrior, UniformPrior,
)
from composed.backends.fsps import FSPSBackend

sfh = ContinuitySFH(
    age="age_fraction",
    age_kind="fraction_of_universe",
    lookback_edges_gyr=(0.0, 0.01, 0.03, 0.1, 0.3),
    samples_per_bin=8,
)

parameters = ParameterSpace(
    names=("zred", "log10_mass", "logzsol", "dust2", "age_fraction", *sfh.ratio_names),
    priors={
        "zred": UniformPrior(0.05, 5.0),
        "log10_mass": UniformPrior(6.0, 13.0),
        "logzsol": UniformPrior(-1.5, 0.3),
        "dust2": UniformPrior(0.0, 2.0),
        "age_fraction": UniformPrior(0.30, 0.95),
        **{name: StudentTPrior(df=2.0, loc=0.0, scale=0.3) for name in sfh.ratio_names},
    },
)

backend = FSPSBackend(
    sfh=sfh,
    sp_kwargs={
        "sfh": 3, "imf_type": 1, "zcontinuous": 1, "dust_type": 2,
        "add_neb_emission": True, "add_igm_absorption": True,
        "add_dust_emission": False,
    },
    default_z_key="zred",
)
MODEL_DISCREPANCY = 0.05
QUICK = os.environ.get("COMPOSED_TUTORIAL_QUICK", "0") == "1"
noise_checkpoint = REPO_ROOT / "outputs" / (
    "tutorial_cosmos2020_survey_noise_quick" if QUICK
    else "tutorial_cosmos2020_survey_noise"
)

if (noise_checkpoint / "manifest.json").exists():
    survey_noise = ConditionalCatalogNoise.load(noise_checkpoint, device="cpu")
    print("Loaded shared COSMOS2020 survey-noise model")
else:
    catalog_magnitude = np.full_like(flux, np.nan)
    positive_flux = np.isfinite(flux) & (flux > 0.0)
    catalog_magnitude[positive_flux] = -2.5 * np.log10(flux[positive_flux])
    survey_noise = ConditionalCatalogNoise.fit(
        catalog_magnitude,
        sigma,
        band_names=BAND_NAMES,
        flux_unit="maggies",
        seed=40,
        hidden_features=64,
        num_transforms=4,
        num_blocks=2,
        epochs=3 if QUICK else 50,
        batch_size=1024,
        validation_split=0.1,
        patience=10,
        device="cpu",
        support_policy="warn",
        invalid_rows="filter",
        catalog_source=str(DATA_PATH),
        row_selection="prepared COSMOS2020 complete ugrizYJH rows",
    )
    survey_noise.save(noise_checkpoint)

# The noise flow is calibrated only on the complete-row COSMOS support.
# Prior draws outside that support are explicitly rejected and resampled.
survey_noise.support_policy = "raise"

data = SEDDataset(
    BAND_NAMES,
    flux[SINGLE_OBJECT],
    sigma[SINGLE_OBJECT],
    flux_unit="maggies",
)
problem = Problem(
    backend,
    parameters,
    data,
    Gaussian(photometric_model_discrepancy=MODEL_DISCREPANCY),
    filters=filters,
)


### Survey noise versus model discrepancy

`sigma` is the raw COSMOS catalog uncertainty and is the uncertainty supplied
to the neural context at both training and inference. `survey_noise` learns the
joint multiband distribution
`q(log10 sigma_catalog | noiseless AB magnitudes)` from complete catalog rows.

The separate `MODEL_DISCREPANCY` enters only through the declared likelihood:

`sigma_draw^2 = sigma_catalog^2 + (MODEL_DISCREPANCY * f_model)^2`.

It is never estimated from `f_obs` and is never added to the SBI context.

The four Student-t ratio priors are the continuity prior. `age_fraction` keeps the
oldest SFH edge below the age of the Universe. `log10_mass` is surviving stellar mass.
Nebular emission and IGM absorption are on; dust emission is omitted because these
bands do not constrain the infrared reradiation.


In [ ]:
from composed import (
    InferenceResult, MAF, PhotometricContext, Simulate, TrainedMAFSBI, fit,
    problem_fingerprint,
)

QUICK = os.environ.get("COMPOSED_TUTORIAL_QUICK", "0") == "1"
N_TRAIN = 2_000 if QUICK else 300_000
N_TARGET = 512 if QUICK else flux.shape[0]
if QUICK and SINGLE_OBJECT >= N_TARGET:
    TARGET_INDICES = np.concatenate([np.arange(N_TARGET - 1), [SINGLE_OBJECT]])
else:
    TARGET_INDICES = np.arange(N_TARGET)
SINGLE_SUMMARY_POSITION = int(np.flatnonzero(TARGET_INDICES == SINGLE_OBJECT)[0])
N_POSTERIOR = 32 if QUICK else 128
N_SINGLE_DRAWS = 512 if QUICK else 30_000
EPOCHS = 10 if QUICK else 200
N_WORKERS = min(8, os.cpu_count() or 1)
OUTPUT = REPO_ROOT / "outputs/tutorial_fsps_maf_cosmos2020"
CHECKPOINT = OUTPUT / "maf"
FORCE = os.environ.get("COMPOSED_TUTORIAL_FORCE", "0") == "1"
OUTPUT.mkdir(parents=True, exist_ok=True)

simulation = Simulate(
    n=N_TRAIN, noise_model=survey_noise, infer=parameters.names,
    context=PhotometricContext("snr_logsigma", flux_unit="maggies"),
    n_workers=N_WORKERS, batch_size=128, executor="process", mp_context="spawn",
    failure_policy="resample", max_retries=max(1000, N_TRAIN),
)

posterior = None
if (CHECKPOINT / "manifest.json").exists() and not FORCE:
    try:
        candidate = TrainedMAFSBI.load(CHECKPOINT, device="auto")
        saved_problem = candidate.metadata["training_set_metadata"]["problem"]
        if problem_fingerprint(saved_problem) != problem_fingerprint(problem):
            raise ValueError("checkpoint was trained for a different Problem")
        posterior = candidate
        print(f"Loaded matching MAF checkpoint: {CHECKPOINT.relative_to(REPO_ROOT)}")
    except (FileNotFoundError, KeyError, RuntimeError, TypeError, ValueError) as error:
        print(f"Ignoring stale MAF checkpoint: {error}")

t0 = time.perf_counter()
if posterior is None:
    maf_single = fit(
        problem,
        MAF(
            hidden_features=256, num_transforms=6, num_blocks=3, epochs=EPOCHS,
            batch_size=2048, validation_split=0.1, patience=20, device="auto",
            num_samples=N_SINGLE_DRAWS, inference_batch_size=8192,
        ),
        training=simulation,
        seed=42,
    )
    posterior = maf_single.inference_state
    posterior.save(CHECKPOINT, overwrite=True)
    print(f"End-to-end SBI fit: {time.perf_counter() - t0:.1f} s")
else:
    single_draws = posterior.sample(
        data, num_samples=N_SINGLE_DRAWS, batch_size=8192, seed=42,
    )[0]
    maf_single = InferenceResult(
        samples=single_draws,
        logp=None,
        weights=np.ones(N_SINGLE_DRAWS),
        parameter_names=posterior.theta_names,
        sampler_name="maf",
        metadata={"problem": problem.specification(), "loaded_checkpoint": str(CHECKPOINT)},
        inference_state=posterior,
    )
    print(f"Checkpoint load and single-object sampling: {time.perf_counter() - t0:.1f} s")
print("MAF device:", posterior.estimator.device)


## Batched catalog inference


In [ ]:
t0 = time.perf_counter()
summary = posterior.summarize_catalog(
    flux[TARGET_INDICES], sigma=sigma[TARGET_INDICES], input_units="native",
    num_samples=N_POSTERIOR, batch_size=8192, seed=43,
)
inference_seconds = time.perf_counter() - t0
summary.save(OUTPUT / "catalog_summary.npz")
print(f"{N_TARGET:,} galaxies x {N_POSTERIOR} draws in {inference_seconds:.2f} s")
print(f"{N_TARGET * N_POSTERIOR / inference_seconds:,.0f} posterior draws/s")


In [ ]:
z_index = posterior.theta_names.index("zred")
mass_index = posterior.theta_names.index("log10_mass")
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].hexbin(z_reference[TARGET_INDICES], summary.median[:, z_index], gridsize=70, mincnt=1)
axes[1].hexbin(mass_reference[TARGET_INDICES], summary.median[:, mass_index], gridsize=70, mincnt=1)
for ax, name in zip(axes, ("redshift", "log10 stellar mass")):
    lo, hi = ax.get_xlim(); ax.plot([lo, hi], [lo, hi], "--", color="0.5")
    ax.set(xlabel=f"COSMOS2020/LePhare {name}", ylabel=f"CompoSED posterior median {name}")
fig.tight_layout()


## MAF posterior for the single-galaxy comparison

This is the same COSMOS2020 galaxy and the same FSPS model fitted with PocoMC in
`04_fsps_pocomc_single_galaxy.ipynb`. We retain 30,000 raw MAF draws here so the
learned posterior geometry can be inspected directly.


In [ ]:
from composed import load_inference_result, require_result_matches_problem
from composed.plot import plot_corner_hexbin

reference_path = REPO_ROOT / "outputs/tutorial_fsps_pocomc_single"
reference_mc = None
if reference_path.exists():
    try:
        reference_mc = require_result_matches_problem(
            load_inference_result(reference_path), problem
        )
    except ValueError as error:
        print(f"Ignoring PocoMC result for a different Problem: {error}")
if reference_mc is None:
    print("Run 04_fsps_pocomc_single_galaxy.ipynb to add the PocoMC contours.")

corner_parameters = ["zred", "log10_mass", "logzsol", "dust2", "age_fraction"]
fig, _ = plot_corner_hexbin(
    maf_single,
    parameters=corner_parameters,
    comparison_result=reference_mc,
    result_label="MAF (300k simulations, 256 x 3)",
    comparison_label="PocoMC",
    max_points=N_SINGLE_DRAWS,
    seed=45,
)
fig.suptitle("FSPS posterior: amortized MAF versus PocoMC", y=1.01)
fig.savefig(OUTPUT / "single_object_maf_vs_pocomc_corner.png", dpi=160, bbox_inches="tight")
print("MAF posterior medians:", dict(zip(posterior.theta_names, maf_single.posterior_median)))
print("LePhare comparison: z=", z_reference[SINGLE_OBJECT], "logM=", mass_reference[SINGLE_OBJECT])


## One physical SFH from the catalog posterior


In [ ]:
from composed import derive_sfh_quantities

median_params = dict(zip(posterior.theta_names, summary.median[SINGLE_SUMMARY_POSITION]))
derived = derive_sfh_quantities(backend, median_params, filters)
lookback = derived.time_gyr[-1] - derived.time_gyr
plt.step(lookback[::-1], derived.sfr_msun_per_yr[::-1], where="mid")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("lookback time [Gyr]"); plt.ylabel("SFR [Msun/yr]")
print("posterior-median log10 SFR:", derived.log10_sfr)
print("LePhare reference log10 SFR:", sfr_reference[SINGLE_OBJECT])
